In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os
from pathlib import Path
import openpyxl
import sqlalchemy as sa

# Desactivar notación científica
pd.set_option('display.float_format', lambda x: '%.3f' % x)
np.set_printoptions(suppress=True)

# Cargar variables de entorno
load_dotenv()

print("✅ Librerías importadas correctamente")


✅ Librerías importadas correctamente


## Fase 1 – Importación

In [2]:
# Definir rutas
data_path = Path('../02_datos/01_Originales')

# Importar Leads.csv
leads_file = data_path / 'Leads.csv'
print(f'Importando {leads_file}...')

df = pd.read_csv(leads_file, sep=';', encoding='utf-8', index_col='id')
print(f'✅ Importación completada')

print(f'Shape: {df.shape}')
df.head()

Importando ../02_datos/01_Originales/Leads.csv...
✅ Importación completada
Shape: (9093, 20)


,origen,fuente,no_enviar_email,no_llamar,compra,visitas_total,tiempo_en_site_total,paginas_vistas_visita,ult_actividad,ambito,ocupacion,conociste_google,conociste_revista,conociste_periodico,conociste_youtube,conociste_facebook,conociste_referencias,score_actividad,score_perfil,descarga_lm
id,,,,,,,,,,,,,,,,,,,,
660737,API,Chat,No,No,0,0.000,0,0.000,Page Visited on Website,Select,Unemployed,No,No,No,No,No,No,15.000,15.000,No
660728,API,Organic Search,No,No,0,5.000,674,2.500,Email Opened,Select,Unemployed,No,No,No,No,No,No,15.000,15.000,No
660727,Landing Page Submission,Direct Traffic,No,No,1,2.000,1532,2.000,Email Opened,Business Administration,Student,No,No,No,No,No,No,14.000,20.000,Yes
660719,Landing Page Submission,Direct Traffic,No,No,0,1.000,305,1.000,Unreachable,Media and Advertising,Unemployed,No,No,No,No,No,No,13.000,17.000,No
660681,Landing Page Submission,Google,No,No,1,2.000,1428,1.000,Converted to Lead,Select,Unemployed,No,No,No,No,No,No,15.000,18.000,No


### Estrategia de integración de datos

Actualmente solo hay una fuente (`Leads.csv`). Si en el futuro se incorporan más fuentes, se recomienda:

- Definir claves de unión (por ejemplo, `id` si está presente en todas).
- Analizar solapamientos y diferencias de granularidad.
- Documentar reglas de prioridad y actualización de registros.
- Realizar pruebas de integridad tras la integración.

¿Quieres dejar documentada alguna integración prevista o pasamos a la siguiente fase?

## Fase 2 – Crear dataset de validacion y train-test

In [ ]:
# Creamos fichero de validación
val = df.sample(frac=0.3)

nombre_fichero_validacion = 'validacion.csv'
ruta_completa_val = '../02_datos/02_Validacion/' + nombre_fichero_validacion

val.to_csv(ruta_completa_val)

In [21]:
# Creamos fichero de trabajo
trabajo = df.loc[~df.index.isin(val.index)]

nombre_fichero_trabajo = 'trabajo.csv'
ruta_completa_trabajo = '../02_datos/03_Entrenamiento/' + nombre_fichero_trabajo

trabajo.to_csv(ruta_completa_trabajo)

## Separación train y test

Se podría extraer una muestra de este dataset pero trabajando con 6365 registros, no tiene sentido disminuir más.

In [3]:
# Separación train/validation sin leakage
from sklearn.model_selection import train_test_split

df_train, df_validation = train_test_split(
    df,
    test_size=0.3,
    random_state=42,
    stratify=None  # Si hay variable objetivo, se puede estratificar
)

print(f"Train shape: {df_train.shape}")
print(f"Validation shape: {df_validation.shape}")

Train shape: (6365, 20)
Validation shape: (2728, 20)


In [4]:
# Guardar los datasets en disco
train_path = "../02_datos/03_Entrenamiento/01_train_tablon_integrado.pkl"
validation_path = "../02_datos/02_Validacion/validation.pkl"
df_train.to_pickle(train_path)
df_validation.to_pickle(validation_path)
print(f"Train guardado en: {train_path}")
print(f"Validation guardado en: {validation_path}")

Train guardado en: ../02_datos/03_Entrenamiento/01_train_tablon_integrado.pkl
Validation guardado en: ../02_datos/02_Validacion/validation.pkl


In [5]:
# Documentar estructura final en copilot-instructions.md
# Mostramos info del dataframe de entrenamiento para documentar
print(df_train.info())

<class 'pandas.DataFrame'>
Index: 6365 entries, 630952 to 592736
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   origen                 6365 non-null   str    
 1   fuente                 6340 non-null   str    
 2   no_enviar_email        6365 non-null   str    
 3   no_llamar              6365 non-null   str    
 4   compra                 6365 non-null   int64  
 5   visitas_total          6273 non-null   float64
 6   tiempo_en_site_total   6365 non-null   int64  
 7   paginas_vistas_visita  6273 non-null   float64
 8   ult_actividad          6297 non-null   str    
 9   ambito                 5353 non-null   str    
 10  ocupacion              4477 non-null   str    
 11  conociste_google       6365 non-null   str    
 12  conociste_revista      6365 non-null   str    
 13  conociste_periodico    6365 non-null   str    
 14  conociste_youtube      6365 non-null   str    
 15  conociste_fac